# Hourly Energy Disaggregation — Cleaned Notebook

Predicts 24-hour profiles of **Grid Import** and **PV Generation** from a
single net-meter input. Three datasets are evaluated in turn:

1. **WP2** — single farm, baseline.
2. **Dairy-farm PVs** — 13 farm/PV pairs concatenated.
3. **PV + Battery** — battery-augmented variant.

Models compared per dataset: **GBRT, CNN, LSTM, BiLSTM, BiLSTM+Attention,
CNN+BiLSTM, Transformer, CNN+Transformer hybrid**.

All trained models are saved under `MODELS_DIR/<dataset>/` and predictions
under `OUTPUTS_DIR/<dataset>/` so nothing has to be retrained for plotting.

### Bugs fixed vs. the original
- Removed ~20 duplicate copies of the metric functions (`mae/ret/r2`).
- Stopped shadowing `sklearn.metrics.mean_absolute_error` inside cells.
- Fixed `pred_lstm = ensure_3d(pred_cnn_bilstm)` typo (was overwriting LSTM with CNN-BiLSTM).
- Fixed `pred_CNN_bilstm` → `pred_cnn_bilstm` capitalization.
- GBRT sections no longer permanently flatten `x_train/y_train/y_test`
  in place (was breaking every NN cell that ran afterwards).
- `relative_error_total` always uses `np.sum(np.abs(y_true))` (negatives
  in net-load otherwise made the denominator near-zero).
- Each dataset uses its own variable suffix (`_wp2`, `_dpv`, `_bat`) so
  cells can be re-run in any order without state leaking between datasets.
- Added per-section model saving (was missing entirely).


## 1. Setup & shared utilities

In [ ]:
# Standard imports used across the whole notebook.
import os
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import calendar

import tensorflow as tf
from tensorflow.keras.models import Model, Sequential, load_model
from tensorflow.keras.layers import (
    Input, Dense, Dropout, Reshape, Flatten, Conv1D,
    LSTM, Bidirectional, Attention, Multiply,
    LayerNormalization, MultiHeadAttention, Add,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.losses import Huber

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor

# Reproducibility (best-effort — full determinism on GPU is not guaranteed).
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)


In [ ]:
# All trained models and predictions are written here.
WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
MODELS_DIR = WORKING_DIR / "models"
OUTPUTS_DIR = WORKING_DIR / "outputs"
FIGURES_DIR = WORKING_DIR / "figures"
for d in (MODELS_DIR, OUTPUTS_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("MODELS_DIR :", MODELS_DIR)
print("OUTPUTS_DIR:", OUTPUTS_DIR)
print("FIGURES_DIR:", FIGURES_DIR)


In [ ]:
# --- Metrics (single definition, used everywhere) ---------------------
def mae(y_true, y_pred):
    """Mean absolute error over flattened arrays."""
    return float(np.mean(np.abs(np.ravel(y_true) - np.ravel(y_pred))))

def ret(y_true, y_pred):
    """Relative error in total: sum |err| / sum |y_true|.
    Uses |y_true| in the denominator so negative net-load doesn't blow it up.
    """
    yt = np.ravel(y_true)
    yp = np.ravel(y_pred)
    denom = np.sum(np.abs(yt))
    return float(np.sum(np.abs(yt - yp)) / denom) if denom else float("nan")

def r2(y_true, y_pred):
    """R^2 over flattened arrays."""
    yt = np.ravel(y_true)
    yp = np.ravel(y_pred)
    ss_tot = np.sum((yt - yt.mean()) ** 2)
    if ss_tot == 0:
        return 0.0
    return float(1.0 - np.sum((yt - yp) ** 2) / ss_tot)

def mape(y_true, y_pred, eps=1e-6):
    """MAPE on the grid channel (index 1). Caps tiny denominators."""
    yt = np.array(y_true)[..., 1]
    yp = np.array(y_pred)[..., 1]
    denom = np.where(np.abs(yt) < eps, eps, yt)
    return float(np.mean(np.abs((yt - yp) / denom)) * 100)

def evaluate_all(y_true, y_pred, label=""):
    """Print and return the standard metric trio."""
    m, e, r = mae(y_true, y_pred), ret(y_true, y_pred), r2(y_true, y_pred)
    if label:
        print(f"[{label}]")
    print(f"  MAE: {m:.2f} Wh | RET: {e:.4f} | R^2: {r:.4f}")
    return {"mae": m, "ret": e, "r2": r}


In [ ]:
# --- Reshape helpers --------------------------------------------------
def ensure_3d(pred, n_hours=24, n_outputs=2):
    """Ensure predictions are (samples, hours, outputs)."""
    pred = np.asarray(pred)
    if pred.ndim == 2 and pred.shape[1] == n_hours * n_outputs:
        return pred.reshape((-1, n_hours, n_outputs))
    return pred

# --- Train/val/test split for the (24h, 2-output) datasets ------------
def split_yearly(x_flat, y_flat, n_years_test_val=2, hours_per_year=8760):
    """Split flat hourly arrays into (train, val, test) of shape
    (samples, 24, 1) for x and (samples, 24, 2) for y. Last `hours_per_year`
    hours are test, the previous `hours_per_year` are validation, the rest
    is train.
    """
    x_flat = np.asarray(x_flat).reshape(-1, 1)
    y_flat = np.asarray(y_flat).reshape(-1, 2)

    cutoff_test = -hours_per_year
    cutoff_val = -2 * hours_per_year

    x_tr = x_flat[:cutoff_val].reshape(-1, 24, 1)
    y_tr = y_flat[:cutoff_val].reshape(-1, 24, 2)
    x_va = x_flat[cutoff_val:cutoff_test].reshape(-1, 24, 1)
    y_va = y_flat[cutoff_val:cutoff_test].reshape(-1, 24, 2)
    x_te = x_flat[cutoff_test:].reshape(-1, 24, 1)
    y_te = y_flat[cutoff_test:].reshape(-1, 24, 2)
    return x_tr, y_tr, x_va, y_va, x_te, y_te


In [ ]:
# --- Save / load helpers ---------------------------------------------
def save_keras(model, name, dataset):
    path = MODELS_DIR / dataset / f"{name}.keras"
    path.parent.mkdir(parents=True, exist_ok=True)
    model.save(path)
    return path

def save_sklearn(estimator, name, dataset):
    path = MODELS_DIR / dataset / f"{name}.pkl"
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(estimator, f)
    return path

def save_predictions(pred, name, dataset):
    """Save a (samples, 24, 2) prediction array as .npy."""
    path = OUTPUTS_DIR / dataset / f"{name}.npy"
    path.parent.mkdir(parents=True, exist_ok=True)
    np.save(path, np.asarray(pred))
    return path

def save_metrics(results, dataset):
    path = OUTPUTS_DIR / dataset / "metrics.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(results, f, indent=2)
    return path


In [ ]:
# --- Standard callbacks for Keras models ------------------------------
def default_callbacks(name, dataset, patience=10):
    ckpt_path = MODELS_DIR / dataset / f"{name}_best.keras"
    ckpt_path.parent.mkdir(parents=True, exist_ok=True)
    return [
        EarlyStopping(monitor="val_loss", patience=patience,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, verbose=1),
        ModelCheckpoint(ckpt_path, monitor="val_loss",
                        save_best_only=True, verbose=0),
    ]


In [ ]:
# --- Plot helpers used by every dataset section -----------------------
DAYS_IN_MONTH = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
MONTH_INDICES = np.repeat(np.arange(12), DAYS_IN_MONTH)

DEFAULT_COLORS = {
    "GBRT": "#1f77b4",
    "CNN": "#ff7f0e",
    "LSTM": "#8c564b",
    "BiLSTM": "#17becf",
    "BiLSTM_Attn": "#bcbd22",
    "CNN_BiLSTM": "#2ca02c",
    "Transformer": "#d62728",
    "CNN_Transformer": "#7f7f7f",
    "Ensemble": "#9467bd",
}

def monthly_metric(predictions, y_true, fn):
    """fn(y_flat, p_flat) -> scalar; returns {model: [12 values]}"""
    out = {name: [] for name in predictions}
    for m in range(12):
        mask = MONTH_INDICES == m
        y_m = y_true[mask]
        for name, p in predictions.items():
            out[name].append(fn(y_m.reshape(-1), p[mask].reshape(-1)))
    return out

def hourly_metric(predictions, y_true, fn):
    out = {name: [] for name in predictions}
    for h in range(24):
        y_h = y_true[:, h, :]
        for name, p in predictions.items():
            out[name].append(fn(y_h.reshape(-1), p[:, h, :].reshape(-1)))
    return out

def bar_plot(values, x_labels, ylabel, title, save_as, figsize=(12, 6)):
    width = 0.8 / max(len(values), 1)
    x = np.arange(len(x_labels))
    fig, ax = plt.subplots(figsize=figsize)
    for i, (name, ys) in enumerate(values.items()):
        ax.bar(x + i * width, ys, width, label=name,
               color=DEFAULT_COLORS.get(name))
    ax.set_xticks(x + width * (len(values) - 1) / 2)
    ax.set_xticklabels(x_labels)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, axis="y", linestyle="--", alpha=0.6)
    ax.legend(loc="upper right", frameon=True)
    plt.tight_layout()
    out = FIGURES_DIR / save_as
    plt.savefig(out)
    plt.show()
    return out

def daily_mae_boxplot(predictions, y_true, save_as):
    daily = {name: np.abs(y_true - p).mean(axis=(1, 2)) for name, p in predictions.items()}
    plt.figure(figsize=(10, 6))
    plt.boxplot(list(daily.values()), labels=list(daily.keys()),
                patch_artist=True, showmeans=True, meanline=True,
                boxprops=dict(facecolor="lightgray", color="black"),
                medianprops=dict(color="black"),
                meanprops=dict(color="red", linestyle="--", linewidth=1.5),
                flierprops=dict(marker="o", markersize=4, alpha=0.3))
    plt.ylabel("Daily MAE (Wh)")
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    out = FIGURES_DIR / save_as
    plt.savefig(out, bbox_inches="tight")
    plt.show()
    return out


In [ ]:
# --- Model factories --------------------------------------------------
def build_cnn(input_shape=(24, 1), num_filters=64, kernel_size=3,
              dense_1=239, dense_2=115, dropout=0.0, lr=2.13e-4):
    model = Sequential([
        Input(shape=input_shape),
        Conv1D(num_filters, kernel_size=kernel_size, activation="relu"),
        Flatten(),
        Dense(dense_1, activation="relu"),
        Dropout(dropout),
        Dense(dense_2, activation="relu"),
        Dense(24 * 2, activation="linear"),
        Reshape((24, 2)),
    ])
    model.compile(optimizer=Adam(learning_rate=lr), loss="mse", metrics=["mae"])
    return model

def build_lstm(input_shape=(24, 1), lstm_units=64, dense_1=239, dense_2=115,
               dropout=0.0, lr=2.13e-4):
    model = Sequential([
        Input(shape=input_shape),
        LSTM(lstm_units, return_sequences=False),
        Dropout(dropout),
        Dense(dense_1, activation="relu"),
        Dense(dense_2, activation="relu"),
        Dense(24 * 2, activation="linear"),
        Reshape((24, 2)),
    ])
    model.compile(optimizer=Adam(learning_rate=lr), loss="mse", metrics=["mae"])
    return model

def build_bilstm(input_shape=(24, 1)):
    inputs = Input(shape=input_shape)
    x = Bidirectional(LSTM(64, return_sequences=True))(inputs)
    x = Dropout(0.2)(x)
    x = Bidirectional(LSTM(64, return_sequences=True))(x)
    x = Dropout(0.2)(x)
    x = Dense(128, activation="relu")(x)
    x = Dropout(0.2)(x)
    outputs = Dense(2, activation="linear")(x)
    model = Model(inputs, outputs)
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

def build_bilstm_attention(input_shape=(24, 1)):
    inputs = Input(shape=input_shape)
    x = Bidirectional(LSTM(64, return_sequences=True))(inputs)
    attn = Attention()([x, x])
    x = Multiply()([x, attn])
    x = LayerNormalization()(x)
    x = Dense(128, activation="relu")(x)
    x = Dropout(0.2)(x)
    outputs = Dense(2, activation="linear")(x)
    model = Model(inputs, outputs)
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

def build_cnn_bilstm(input_shape=(24, 1)):
    inputs = Input(shape=input_shape)
    x = Conv1D(64, 3, activation="relu", padding="same")(inputs)
    x = Conv1D(64, 3, activation="relu", padding="same")(x)
    x = Bidirectional(LSTM(64, return_sequences=True))(x)
    x = LayerNormalization()(x)
    x = Dense(128, activation="relu")(x)
    x = Dropout(0.2)(x)
    outputs = Dense(2, activation="linear")(x)
    model = Model(inputs, outputs)
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

def transformer_encoder(inputs, head_size=64, num_heads=4, ff_dim=128, dropout=0.1):
    x = MultiHeadAttention(key_dim=head_size, num_heads=num_heads)(inputs, inputs)
    x = Dropout(dropout)(x)
    x = LayerNormalization(epsilon=1e-6)(x)
    res = Add()([x, inputs])
    x = Dense(ff_dim, activation="relu")(res)
    x = Dropout(dropout)(x)
    x = Dense(inputs.shape[-1])(x)
    x = LayerNormalization(epsilon=1e-6)(x)
    return Add()([x, res])

def build_transformer(input_shape=(24, 1), head_size=32, num_heads=8,
                      ff_dim=212, dense_1=222, dense_2=102,
                      dropout=0.349, lr=1.49e-3, stack=1):
    inputs = Input(shape=input_shape)
    x = inputs
    for _ in range(stack):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)
    x = Flatten()(x)
    x = Dense(dense_1, activation="relu")(x)
    x = Dense(dense_2, activation="relu")(x)
    x = Dense(24 * 2, activation="linear")(x)
    outputs = Reshape((24, 2))(x)
    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=lr), loss="mse", metrics=["mae"])
    return model

def build_cnn_transformer(input_shape=(24, 1)):
    inputs = Input(shape=input_shape)
    x = Conv1D(64, 3, activation="relu", padding="same")(inputs)
    x = Conv1D(64, 3, activation="relu", padding="same")(x)
    x = transformer_encoder(x, head_size=32, num_heads=4, ff_dim=128, dropout=0.1)
    x = Dense(128, activation="relu",
              kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = Dropout(0.2)(x)
    outputs = Dense(2, activation="linear")(x)
    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=1e-3), loss="mse", metrics=["mae"])
    return model


In [ ]:
# --- The training driver used by every dataset section ----------------
def train_keras(model_name, builder, dataset, x_tr, y_tr, x_va, y_va,
                epochs=60, batch_size=32):
    print(f"\n>>> Training {model_name} on {dataset}")
    model = builder()
    history = model.fit(
        x_tr, y_tr,
        validation_data=(x_va, y_va),
        epochs=epochs, batch_size=batch_size,
        callbacks=default_callbacks(model_name, dataset),
        verbose=1,
    )
    save_keras(model, model_name, dataset)
    return model, history

def train_gbrt(dataset, x_tr, y_tr, n_estimators=200, lr=0.1, max_depth=5):
    """Train GBRT without mutating the caller's arrays."""
    print(f"\n>>> Training GBRT on {dataset}")
    x_tr_flat = x_tr.reshape((x_tr.shape[0], -1))
    y_tr_flat = y_tr.reshape((y_tr.shape[0], -1))
    model = MultiOutputRegressor(GradientBoostingRegressor(
        n_estimators=n_estimators, learning_rate=lr,
        max_depth=max_depth, random_state=SEED,
    ))
    model.fit(x_tr_flat, y_tr_flat)
    save_sklearn(model, "gbrt", dataset)
    return model

def predict_gbrt(model, x_te):
    """Predict and reshape to (n, 24, 2) without mutating x_te."""
    flat = model.predict(x_te.reshape((x_te.shape[0], -1)))
    return flat.reshape((-1, 24, 2))


In [ ]:
# --- Run-all-models helper -------------------------------------------
def train_all_models(dataset, x_tr, y_tr, x_va, y_va, x_te, y_te,
                     epochs=60, include=None):
    """Train every model on a dataset, save it, save predictions, and
    return (predictions_dict, metrics_dict). `include` filters the model list."""
    builders = {
        "GBRT":           None,  # special-cased
        "CNN":            build_cnn,
        "LSTM":           build_lstm,
        "BiLSTM":         build_bilstm,
        "BiLSTM_Attn":    build_bilstm_attention,
        "CNN_BiLSTM":     build_cnn_bilstm,
        "Transformer":    build_transformer,
        "CNN_Transformer":build_cnn_transformer,
    }
    if include is not None:
        builders = {k: v for k, v in builders.items() if k in include}

    predictions, metrics = {}, {}

    if "GBRT" in builders:
        gbrt = train_gbrt(dataset, x_tr, y_tr)
        pred = predict_gbrt(gbrt, x_te)
        predictions["GBRT"] = pred
        save_predictions(pred, "GBRT", dataset)
        metrics["GBRT"] = evaluate_all(y_te, pred, f"GBRT/{dataset}")

    for name, builder in builders.items():
        if name == "GBRT":
            continue
        model, _ = train_keras(name, builder, dataset, x_tr, y_tr, x_va, y_va,
                               epochs=epochs)
        pred = ensure_3d(model.predict(x_te))
        predictions[name] = pred
        save_predictions(pred, name, dataset)
        metrics[name] = evaluate_all(y_te, pred, f"{name}/{dataset}")

    save_metrics(metrics, dataset)
    return predictions, metrics


## 2. Dataset 1 — Work Package 2 (single farm)

Net-meter input → (Grid Import, PV Generation) per hour. 4015 train days,
365 validation days, 365 test days.

In [ ]:
DATASET_WP2 = "wp2"

# Try Kaggle path first, then a local fallback.
_kaggle_x = "/kaggle/input/work-package-2/x_flattened.csv"
_kaggle_y = "/kaggle/input/work-package-2/y_flattened.csv"
_local_x = "/Users/hosseinkhaleghy/Desktop/files/wp2/work package 2/x_flattened.csv"
_local_y = "/Users/hosseinkhaleghy/Desktop/files/wp2/work package 2/y_flattened.csv"

x_path = _kaggle_x if os.path.exists(_kaggle_x) else _local_x
y_path = _kaggle_y if os.path.exists(_kaggle_y) else _local_y
print("Loading WP2:", x_path, "/", y_path)

x_wp2_raw = pd.read_csv(x_path)["meter"].to_numpy()
y_wp2_raw = pd.read_csv(y_path, index_col="Unnamed: 0").to_numpy()

(x_train_wp2, y_train_wp2,
 x_val_wp2,   y_val_wp2,
 x_test_wp2,  y_test_wp2) = split_yearly(x_wp2_raw, y_wp2_raw)

print("shapes:", x_train_wp2.shape, y_train_wp2.shape,
      x_val_wp2.shape, y_test_wp2.shape)


In [ ]:
# Quick visual sanity check on one training day
plt.figure(figsize=(10, 4))
plt.plot(x_train_wp2[1, :, 0], label="net meter")
plt.plot(y_train_wp2[1, :, 0], label="Grid Import")
plt.plot(y_train_wp2[1, :, 1], label="PV Generation")
plt.xlabel("Hour"); plt.ylabel("Wh")
plt.legend(); plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "wp2_sample_day.pdf")
plt.show()


In [ ]:
# Train every model and save weights + predictions.
preds_wp2, metrics_wp2 = train_all_models(
    DATASET_WP2,
    x_train_wp2, y_train_wp2,
    x_val_wp2,   y_val_wp2,
    x_test_wp2,  y_test_wp2,
    epochs=60,
)


In [ ]:
# If you want to skip retraining, this loads the saved predictions:
# preds_wp2 = {p.stem: np.load(p) for p in (OUTPUTS_DIR / DATASET_WP2).glob("*.npy")}

# Monthly + hourly comparison
monthly_mae_wp2 = monthly_metric(preds_wp2, y_test_wp2, lambda y, p: np.mean(np.abs(y - p)))
hourly_mae_wp2  = hourly_metric(preds_wp2, y_test_wp2, lambda y, p: np.mean(np.abs(y - p)))

bar_plot(monthly_mae_wp2,
         [calendar.month_abbr[i + 1] for i in range(12)],
         "MAE (Wh)", "WP2 — Monthly MAE", "wp2_monthly_mae.pdf")
bar_plot(hourly_mae_wp2,
         [str(h) for h in range(24)],
         "MAE (Wh)", "WP2 — Hourly MAE", "wp2_hourly_mae.pdf",
         figsize=(16, 6))

daily_mae_boxplot(preds_wp2, y_test_wp2, "wp2_daily_mae_boxplot.pdf")


In [ ]:
# Per-day overlay of all models for one example day (test day #80)
day = 80
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharex=True)
for ax, ch, title in [(axes[0], 0, "Grid Import"), (axes[1], 1, "PV Generation")]:
    for name, p in preds_wp2.items():
        ax.plot(p[day, :, ch], linestyle="--", label=name,
                color=DEFAULT_COLORS.get(name))
    ax.plot(y_test_wp2[day, :, ch], color="black", linewidth=2, label="Ground truth")
    ax.set_title(title); ax.set_xlabel("Hour"); ax.set_ylabel("Wh")
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "wp2_day_overlay.pdf")
plt.show()


## 3. Dataset 2 — Different PVs (Dairy farm aggregation)

13 farms × 1 year × hourly data. For each farm we compute
`net = farm_load - PV` and stack vertically so models see one long
sequence of (`net → (load, PV)`) pairs.

In [ ]:
DATASET_DPV = "different_pvs"

import glob

# Locate Kaggle inputs or local fallbacks.
_kaggle_pv = sorted(glob.glob("/kaggle/input/dairy-farm-pvs/PV*.csv"))
_local_pv  = sorted(glob.glob("/Users/hosseinkhaleghy/Desktop/files/wp2/Dairy farm PVs/PV*.csv"))
csv_files_pv = _kaggle_pv or _local_pv

_kaggle_uni = "/kaggle/input/dataset-unisex1/dataset_final_unix.csv"
_local_uni  = "/Users/hosseinkhaleghy/Desktop/files/wp2/dataset_final_unix/dataset_final_unix.csv"
unix_path = _kaggle_uni if os.path.exists(_kaggle_uni) else _local_uni

print("PV files :", len(csv_files_pv), "found")
print("Unix file:", unix_path)


In [ ]:
# Build per-farm PV dataframe, scale to Wh, then strip noise around zero.
pv_frames = []
for f in csv_files_pv:
    df = pd.read_csv(f, index_col=0)
    col = Path(f).stem[-3:]                    # last 3 chars of file name
    df.rename(columns={df.columns[0]: col}, inplace=True)
    df.loc[(df[col] > -0.01) & (df[col] < 0.01), col] = 0
    pv_frames.append(df)

dataset_pv = (pd.concat(pv_frames, axis=1).iloc[:8760]) * 1000  # to Wh

dataset_uni = pd.read_csv(unix_path, index_col="Unnamed: 0")
for c in ("timestamp", "SAM"):
    if c in dataset_uni.columns:
        dataset_uni.drop(columns=c, inplace=True)

farm_cols = list(dataset_uni.columns[dataset_uni.columns.str.match(r".*\d$")])
data_farm = dataset_uni[farm_cols]

dataset_pv = dataset_pv.reset_index(drop=True)
if "Time stamp" in dataset_pv.columns:
    dataset_pv.drop(columns="Time stamp", inplace=True)

dataset_combined = pd.concat([data_farm, dataset_pv], axis=1)
print("combined shape:", dataset_combined.shape)


In [ ]:
# For every farm i, compute Diff_i = farm_i - PV_i (or V_i fallback).
diff_frame = pd.DataFrame()
paired_data = []
missing = []

for i in range(1, 15):
    farm_col = f"farm{i}"
    pv_col = next(
        (c for c in dataset_combined.columns if c in (f"PV{i}", f"V{i}")),
        None,
    )
    if pv_col and farm_col in dataset_combined.columns:
        diff_frame[f"Diff{i}"] = dataset_combined[farm_col] - dataset_combined[pv_col]
        paired_data.append(np.column_stack(
            (dataset_combined[farm_col].to_numpy(),
             dataset_combined[pv_col].to_numpy())
        ))
    else:
        missing.append((farm_col, pv_col))

if missing:
    print("Missing pairs:", missing)

x_dpv_flat = diff_frame.values.flatten("F").reshape(-1, 1)  # (8760 * n_farms, 1)
y_dpv_flat = np.vstack(paired_data)                          # (8760 * n_farms, 2)
print("flat shapes:", x_dpv_flat.shape, y_dpv_flat.shape)

# Persist the aggregated per-farm dataset
pd.DataFrame(x_dpv_flat).to_csv(OUTPUTS_DIR / DATASET_DPV / "x_diff.csv", index=False)
pd.DataFrame(y_dpv_flat, columns=["load", "pv"]).to_csv(
    OUTPUTS_DIR / DATASET_DPV / "y_pairs.csv", index=False)


In [ ]:
(x_train_dpv, y_train_dpv,
 x_val_dpv,   y_val_dpv,
 x_test_dpv,  y_test_dpv) = split_yearly(x_dpv_flat, y_dpv_flat)

print("split shapes:",
      x_train_dpv.shape, y_train_dpv.shape,
      x_val_dpv.shape, y_test_dpv.shape)


In [ ]:
preds_dpv, metrics_dpv = train_all_models(
    DATASET_DPV,
    x_train_dpv, y_train_dpv,
    x_val_dpv,   y_val_dpv,
    x_test_dpv,  y_test_dpv,
    epochs=60,
)


In [ ]:
monthly_mae_dpv = monthly_metric(preds_dpv, y_test_dpv,
                                 lambda y, p: np.mean(np.abs(y - p)))
hourly_mae_dpv  = hourly_metric(preds_dpv, y_test_dpv,
                                 lambda y, p: np.mean(np.abs(y - p)))

bar_plot(monthly_mae_dpv,
         [calendar.month_abbr[i + 1] for i in range(12)],
         "MAE (Wh)", "Different PVs — Monthly MAE", "dpv_monthly_mae.pdf")
bar_plot(hourly_mae_dpv,
         [str(h) for h in range(24)],
         "MAE (Wh)", "Different PVs — Hourly MAE", "dpv_hourly_mae.pdf",
         figsize=(16, 6))
daily_mae_boxplot(preds_dpv, y_test_dpv, "dpv_daily_mae_boxplot.pdf")


## 4. Dataset 3 — PV + Battery

Per-farm storage-augmented variant. For each farm i: `diff_i = farm_unix_i - pv_battery_i`
(the battery scales the PV signature). Outputs are (`farm_unix_i`, `pv_battery_i*1000`).

In [ ]:
DATASET_BAT = "pv_battery"

import re

_kaggle_bat = sorted(glob.glob("/kaggle/input/pv-battery/pv-bat*.csv"))
_local_bat  = sorted(glob.glob("/Users/hosseinkhaleghy/Desktop/files/wp2/pv-battery/pv-bat*.csv"))
csv_files_bat = _kaggle_bat or _local_bat

print("PV-battery files:", len(csv_files_bat), "found")

bat_frames = []
for f in csv_files_bat:
    df = pd.read_csv(f, index_col=0)
    col = Path(f).stem[-3:]
    df.rename(columns={df.columns[0]: col}, inplace=True)
    df.loc[(df[col] > -0.01) & (df[col] < 0.01), col] = 0
    bat_frames.append(df)

dataset_bat = pd.concat(bat_frames, axis=1).reset_index(drop=True).iloc[:8760]

# Reload the unix farm dataset (kept separate from the DPV section).
dataset_uni_bat = pd.read_csv(unix_path, index_col="Unnamed: 0")
for c in ("timestamp", "SAM"):
    if c in dataset_uni_bat.columns:
        dataset_uni_bat.drop(columns=c, inplace=True)

farm_only = dataset_uni_bat.loc[
    :, dataset_uni_bat.columns.str.contains(r"\d$")
]


In [ ]:
def col_number(name):
    digits = "".join(filter(str.isdigit, name.split()[-1]))
    return int(digits) if digits else float("inf")

new_cols = {col_number(c): c for c in farm_only.columns}
old_cols = {col_number(c): c for c in dataset_bat.columns}
shared_keys = sorted(set(new_cols) & set(old_cols))

# Per-farm net = farm_unix - pv_battery (matched by trailing index).
diff = pd.DataFrame()
for k in shared_keys:
    diff[f"diff_{k}"] = farm_only[new_cols[k]] - dataset_bat[old_cols[k]]

# Sort battery columns by trailing index so y_2 alignment is correct.
sorted_bat = dataset_bat[sorted(dataset_bat.columns, key=col_number)]

x_bat_flat = diff.values.flatten("F").reshape(-1, 1)
y_bat1 = farm_only[[new_cols[k] for k in shared_keys]].values.flatten("F").reshape(-1, 1)
y_bat2 = sorted_bat.values.flatten("F").reshape(-1, 1) * 1000
y_bat_flat = np.hstack([y_bat1, y_bat2])

print("flat shapes:", x_bat_flat.shape, y_bat_flat.shape)

pd.DataFrame(x_bat_flat).to_csv(OUTPUTS_DIR / DATASET_BAT / "x_bat.csv", index=False)
pd.DataFrame(y_bat_flat, columns=["load", "pv"]).to_csv(
    OUTPUTS_DIR / DATASET_BAT / "y_bat.csv", index=False)


In [ ]:
(x_train_bat, y_train_bat,
 x_val_bat,   y_val_bat,
 x_test_bat,  y_test_bat) = split_yearly(x_bat_flat, y_bat_flat)

preds_bat, metrics_bat = train_all_models(
    DATASET_BAT,
    x_train_bat, y_train_bat,
    x_val_bat,   y_val_bat,
    x_test_bat,  y_test_bat,
    epochs=60,
)


In [ ]:
monthly_mae_bat = monthly_metric(preds_bat, y_test_bat,
                                 lambda y, p: np.mean(np.abs(y - p)))
hourly_mae_bat  = hourly_metric(preds_bat, y_test_bat,
                                 lambda y, p: np.mean(np.abs(y - p)))

bar_plot(monthly_mae_bat,
         [calendar.month_abbr[i + 1] for i in range(12)],
         "MAE (Wh)", "PV+Battery — Monthly MAE", "bat_monthly_mae.pdf")
bar_plot(hourly_mae_bat,
         [str(h) for h in range(24)],
         "MAE (Wh)", "PV+Battery — Hourly MAE", "bat_hourly_mae.pdf",
         figsize=(16, 6))
daily_mae_boxplot(preds_bat, y_test_bat, "bat_daily_mae_boxplot.pdf")


## 5. Hourly best-model ensemble

Pick the per-hour MAE-winner across all candidate models and stitch its
predictions into a single 24h profile. Done here for the WP2 dataset; the
helper is dataset-agnostic.

In [ ]:
def hourly_best_ensemble(predictions, y_true):
    """For each hour, pick the model with the smallest MAE and assemble
    predictions from those winners."""
    names = list(predictions.keys())
    stacked = np.stack([predictions[n] for n in names], axis=0)  # (M, N, 24, 2)

    hourly_mae = np.zeros((len(names), 24))
    for m, n in enumerate(names):
        for h in range(24):
            hourly_mae[m, h] = np.mean(np.abs(stacked[m, :, h, :] - y_true[:, h, :]))
    best = np.argmin(hourly_mae, axis=0)

    ensemble = np.zeros_like(y_true)
    for h in range(24):
        ensemble[:, h, :] = stacked[best[h], :, h, :]
    return ensemble, [names[i] for i in best]

ensemble_wp2, picks_wp2 = hourly_best_ensemble(preds_wp2, y_test_wp2)
print("Hourly winners (WP2):", picks_wp2)

evaluate_all(y_test_wp2, ensemble_wp2, "Ensemble/wp2")
save_predictions(ensemble_wp2, "Ensemble", DATASET_WP2)


In [ ]:
# Add ensemble into the comparison plots.
preds_wp2_with_ens = {**preds_wp2, "Ensemble": ensemble_wp2}
monthly_mae_wp2_e = monthly_metric(preds_wp2_with_ens, y_test_wp2,
                                   lambda y, p: np.mean(np.abs(y - p)))
bar_plot(monthly_mae_wp2_e,
         [calendar.month_abbr[i + 1] for i in range(12)],
         "MAE (Wh)", "WP2 + Ensemble — Monthly MAE",
         "wp2_monthly_mae_ensemble.pdf")
daily_mae_boxplot(preds_wp2_with_ens, y_test_wp2,
                  "wp2_daily_mae_boxplot_ensemble.pdf")


## 6. Statistical comparison

Paired t-test of per-sample absolute errors between two models on each
output channel (PV and Grid). One-shot helper so it can be reused across
datasets / model pairs.

In [ ]:
from scipy.stats import ttest_rel

def paired_t(name_a, pred_a, name_b, pred_b, y_true, alpha=0.05):
    err_a = np.abs(pred_a - y_true)
    err_b = np.abs(pred_b - y_true)
    print(f"\nPaired t-test — {name_a} vs {name_b}")
    for ch, label in [(0, "Grid"), (1, "PV")]:
        t, p = ttest_rel(err_a[..., ch].ravel(), err_b[..., ch].ravel())
        verdict = "significant" if p < alpha else "not significant"
        print(f"  {label:5s}: t = {t:+.3f}, p = {p:.2e}  ({verdict})")

paired_t("Ensemble", ensemble_wp2,
         "Transformer", preds_wp2["Transformer"],
         y_test_wp2)
paired_t("Ensemble", ensemble_wp2,
         "CNN_BiLSTM", preds_wp2["CNN_BiLSTM"],
         y_test_wp2)


## 7. Optuna tuning (optional)

Used to pick the hyperparameters baked into the `build_*` factories above.
This section is wrapped in a flag so it doesn't auto-run during a full
notebook execution.

In [ ]:
RUN_OPTUNA = False  # flip to True to retune

if RUN_OPTUNA:
    import optuna
    from sklearn.model_selection import train_test_split

    def cnn_objective(trial, x_tr, y_tr):
        x_a, x_b, y_a, y_b = train_test_split(x_tr, y_tr,
                                              test_size=0.2,
                                              random_state=SEED)
        model = build_cnn(
            num_filters=trial.suggest_categorical("num_filters", [32, 64, 128]),
            kernel_size=trial.suggest_int("kernel_size", 2, 5),
            dense_1=trial.suggest_int("dense_1", 64, 256),
            dense_2=trial.suggest_int("dense_2", 32, 128),
            dropout=trial.suggest_float("dropout", 0.0, 0.4),
            lr=trial.suggest_float("lr", 1e-4, 1e-2, log=True),
        )
        model.fit(x_a, y_a, validation_data=(x_b, y_b),
                  epochs=50, batch_size=32, verbose=0,
                  callbacks=[EarlyStopping(monitor="val_loss", patience=5,
                                            restore_best_weights=True)])
        loss, _ = model.evaluate(x_b, y_b, verbose=0)
        return loss

    study = optuna.create_study(direction="minimize",
                                sampler=optuna.samplers.TPESampler())
    study.optimize(lambda t: cnn_objective(t, x_train_wp2, y_train_wp2),
                   n_trials=30)
    print("CNN best:", study.best_trial.params)


## 8. Saved artifacts

Everything under `models/` and `outputs/` is the result of running the
sections above. Predictions are stored as `.npy` so they round-trip
without quantization; metrics are JSON.

In [ ]:
def show_tree(root, indent=0):
    root = Path(root)
    if not root.exists():
        print(root, "(missing)")
        return
    for p in sorted(root.iterdir()):
        print("  " * indent + p.name + ("/" if p.is_dir() else ""))
        if p.is_dir():
            show_tree(p, indent + 1)

print("== models ==");  show_tree(MODELS_DIR)
print("\n== outputs =="); show_tree(OUTPUTS_DIR)
print("\n== figures =="); show_tree(FIGURES_DIR)
